In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install torch torchvision

In [5]:
import os

# Check plantvillage
plantvillage_path = '/content/drive/MyDrive/AgriAI/datasets/plantvillage'
soil_path = '/content/drive/MyDrive/AgriAI/datasets/soil_types'

print("PlantVillage folders:")
print(os.listdir(plantvillage_path))

print("\nSoil type folders:")
print(os.listdir(soil_path))

PlantVillage folders:
['Tomato___healthy', 'Tomato___Target_Spot', 'Tomato___Septoria_leaf_spot', 'Tomato___Tomato_mosaic_virus', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Leaf_Mold', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Late_blight', 'Tomato___Early_blight', 'Tomato___Bacterial_spot', 'Strawberry___Leaf_scorch', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Pepper,_bell___Bacterial_spot', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Raspberry___healthy', 'Grape___Black_rot', 'Pepper,_bell___healthy', 'Potato___healthy', 'Peach___healthy', 'Peach___Bacterial_spot', 'Potato___Late_blight', 'Corn_(maize)___Northern_Leaf_Blight', 'Grape___Esca_(Black_Measles)', 'Corn_(maize)___Common_rust_', 'Potato___Early_blight', 'Corn_(maize)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Cherry_(including_sour)___Powdery_mildew', 'Blueberry___healthy', 'Cherr

In [6]:
import torch
import torchvision.models as models
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import json

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load PlantVillage dataset
plantvillage_path = '/content/drive/MyDrive/AgriAI/datasets/plantvillage'
dataset = datasets.ImageFolder(plantvillage_path, transform=transform)

# Split 80% train, 20% test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_data, test_data = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

train_loader = DataLoader(train_data, batch_size=32,
                          shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=32,
                         num_workers=2)

print(f"Total images: {len(dataset)}")
print(f"Total classes: {len(dataset.classes)}")
print(f"Train images: {train_size}")
print(f"Test images: {test_size}")
print(f"Classes: {dataset.classes}")

Total images: 54305
Total classes: 38
Train images: 43444
Test images: 10861
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Sep

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")
# Should print: Training on: cuda
# If it says cpu, your GPU is not enabled — go back and set T4 GPU

# Load EfficientNet-B3 with pretrained ImageNet weights
disease_model = models.efficientnet_b3(pretrained=True)

# Replace final layer with our number of disease classes
disease_model.classifier[1] = nn.Linear(
    disease_model.classifier[1].in_features,
    len(dataset.classes)
)
disease_model = disease_model.to(device)

# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(disease_model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=2, gamma=0.5
)

print("Starting disease model training...")
print("This will take approximately 30-60 minutes on T4 GPU")

for epoch in range(5):
    disease_model.train()
    running_loss = 0
    correct = 0
    total = 0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = disease_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Show progress every 50 batches
        if (i + 1) % 50 == 0:
            print(f"  Epoch {epoch+1} | Batch {i+1} | "
                  f"Loss: {running_loss/(i+1):.4f} | "
                  f"Acc: {100*correct/total:.2f}%")

    scheduler.step()
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/5] Complete — "
          f"Accuracy: {epoch_acc:.2f}%")

print("Disease model training complete!")

Training on: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B3_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 80.6MB/s]


Starting disease model training...
This will take approximately 30-60 minutes on T4 GPU
  Epoch 1 | Batch 50 | Loss: 1.5494 | Acc: 64.25%
  Epoch 1 | Batch 100 | Loss: 0.9966 | Acc: 75.66%
  Epoch 1 | Batch 150 | Loss: 0.7918 | Acc: 79.88%
  Epoch 1 | Batch 200 | Loss: 0.6535 | Acc: 83.22%
  Epoch 1 | Batch 250 | Loss: 0.5759 | Acc: 84.80%
  Epoch 1 | Batch 300 | Loss: 0.5200 | Acc: 86.12%
  Epoch 1 | Batch 350 | Loss: 0.4690 | Acc: 87.40%
  Epoch 1 | Batch 400 | Loss: 0.4353 | Acc: 88.18%
  Epoch 1 | Batch 450 | Loss: 0.4031 | Acc: 88.94%
  Epoch 1 | Batch 500 | Loss: 0.3798 | Acc: 89.52%
  Epoch 1 | Batch 550 | Loss: 0.3585 | Acc: 90.02%
  Epoch 1 | Batch 600 | Loss: 0.3388 | Acc: 90.48%
  Epoch 1 | Batch 650 | Loss: 0.3256 | Acc: 90.82%
  Epoch 1 | Batch 700 | Loss: 0.3109 | Acc: 91.24%
  Epoch 1 | Batch 750 | Loss: 0.2984 | Acc: 91.57%
  Epoch 1 | Batch 800 | Loss: 0.2859 | Acc: 91.89%
  Epoch 1 | Batch 850 | Loss: 0.2754 | Acc: 92.14%
  Epoch 1 | Batch 900 | Loss: 0.2653 | Acc: 92

In [8]:
save_path = '/content/drive/MyDrive/AgriAI/models/'
os.makedirs(save_path, exist_ok=True)

# Save model weights
torch.save(disease_model.state_dict(),
           save_path + 'disease_model.pth')

# Save class names — very important for predictions
with open(save_path + 'class_names.json', 'w') as f:
    json.dump(dataset.classes, f)

print("disease_model.pth saved to Google Drive!")
print("class_names.json saved to Google Drive!")
print(f"Total classes saved: {len(dataset.classes)}")

disease_model.pth saved to Google Drive!
class_names.json saved to Google Drive!
Total classes saved: 38
